# Pooled Model Comparison Across Datasets

This notebook pools model comparison results across multiple datasets using:
1. Summed AIC (total AIC across all subjects in pooled datasets)
2. Rank-based aggregation (mean rank across datasets)
3. Meta-analytic pooling of ΔAIC (inverse-variance weighted)

In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy import stats
from jaxcmr.helpers import find_project_root

In [2]:
# Configuration
fit_tag = "rerun_best_of_1"
fit_dir = "projects/repfr/results/fits/"

# Dataset groupings
free_recall_datasets = ["LohnasKahana2014"] #, "Lohnas2025"]
serial_recall_datasets = ["RepeatedRecallsKahanaJacobs2000"] # "RepeatedRecallsGordonRanschburg2021", 
all_datasets = free_recall_datasets + serial_recall_datasets

model_names = [
    "WeirdCMRNoStop",
    "NoReinstateCMRNoStop",
    "DistinctContextsCMRNoStop",
    "BasePositionalCMRNoStop",
    # "FullPositionalCMRNoStop",
    "McfReinfPositionalCMRNoStop",
    # "MfcReinfPositionalCMRNoStop",
    "FullReinfPositionalCMRNoStop",
    # "SimpleFullReinfPositionalCMRNoStop",
    "BlendPositionalCMRNoStop",
]

model_titles = [
    "WeirdCMR",
    "NoReinstateCMR",
    "DistinctContextsCMR",
    "BasePositionalCMR",
    # "FullPositionalCMR",
    "MCFReinfPositionalCMR",
    # "MFCReinfPositionalCMR",
    "FullReinfPositionalCMR",
    # "SimpleFullReinfPositionalCMR",
    "BlendPositionalCMR",
]

In [3]:
def load_results(data_name, model_name):
    """Load fitting results for a dataset-model pair."""
    fit_path = os.path.join(
        find_project_root(), fit_dir,
        f"{data_name}_{model_name}_{fit_tag}.json"
    )
    with open(fit_path) as f:
        result = json.load(f)
    return result

def get_aic_per_subject(result):
    """Compute AIC for each subject: AIC = 2k + 2*NLL."""
    fitness = np.array(result["fitness"])  # NLL per subject (top-level key)
    k = len(result["free"])  # number of free parameters
    return 2 * k + 2 * fitness

def get_total_aic(result):
    """Compute total AIC across all subjects."""
    return np.sum(get_aic_per_subject(result))

def get_mean_aic(result):
    """Compute mean AIC per subject."""
    return np.mean(get_aic_per_subject(result))

In [4]:
# Load all results
all_results = {}
for data_name in all_datasets:
    all_results[data_name] = {}
    for model_name, model_title in zip(model_names, model_titles):
        all_results[data_name][model_title] = load_results(data_name, model_name)

## Method 1: Summed AIC Across Datasets

In [5]:
def compute_summed_aic(datasets, results_dict):
    """Compute total summed AIC across specified datasets for each model."""
    summed_aic = {}
    for model_title in model_titles:
        total = 0
        for data_name in datasets:
            total += get_total_aic(results_dict[data_name][model_title])
        summed_aic[model_title] = total
    return summed_aic

def summed_aic_table(datasets, results_dict, label=""):
    """Create a table of summed AIC results."""
    summed = compute_summed_aic(datasets, results_dict)
    df = pd.DataFrame({
        "Model": list(summed.keys()),
        "Summed AIC": list(summed.values())
    })
    df = df.sort_values("Summed AIC").reset_index(drop=True)
    min_aic = df["Summed AIC"].min()
    df["ΔAIC"] = df["Summed AIC"] - min_aic
    df["Relative Likelihood"] = np.exp(-0.5 * df["ΔAIC"])
    df["AIC Weight"] = df["Relative Likelihood"] / df["Relative Likelihood"].sum()
    return df

In [6]:
print("=" * 60)
print("SUMMED AIC: FREE RECALL DATASETS")
print("=" * 60)
print(f"Datasets: {free_recall_datasets}\n")
df_free = summed_aic_table(free_recall_datasets, all_results)
print(df_free.to_string(index=False))

SUMMED AIC: FREE RECALL DATASETS
Datasets: ['LohnasKahana2014']

                 Model    Summed AIC       ΔAIC  Relative Likelihood    AIC Weight
FullReinfPositionalCMR 107470.293457   0.000000         1.000000e+00  9.999996e-01
    BlendPositionalCMR 107499.887817  29.594360         3.746850e-07  3.746849e-07
 MCFReinfPositionalCMR 107541.019775  70.726318         4.385056e-16  4.385054e-16
     BasePositionalCMR 107582.535156 112.241699         4.236671e-25  4.236669e-25
   DistinctContextsCMR 107623.735352 153.441895         4.791991e-34  4.791989e-34
              WeirdCMR 108030.615845 560.322388        2.125884e-122 2.125883e-122
        NoReinstateCMR 108254.476562 784.183105        5.209574e-171 5.209572e-171


In [7]:
print("=" * 60)
print("SUMMED AIC: SERIAL RECALL DATASETS")
print("=" * 60)
print(f"Datasets: {serial_recall_datasets}\n")
df_serial = summed_aic_table(serial_recall_datasets, all_results)
print(df_serial.to_string(index=False))

SUMMED AIC: SERIAL RECALL DATASETS
Datasets: ['RepeatedRecallsKahanaJacobs2000']

                 Model    Summed AIC         ΔAIC  Relative Likelihood   AIC Weight
FullReinfPositionalCMR 109656.208130     0.000000         1.000000e+00 5.278033e-01
     BasePositionalCMR 109656.430786     0.222656         8.946451e-01 4.721967e-01
    BlendPositionalCMR 109699.987549    43.779419         3.114727e-10 1.643963e-10
 MCFReinfPositionalCMR 109766.738281   110.530151         9.969711e-25 5.262047e-25
        NoReinstateCMR 113025.575928  3369.367798         0.000000e+00 0.000000e+00
              WeirdCMR 113080.492065  3424.283936         0.000000e+00 0.000000e+00
   DistinctContextsCMR 122096.750610 12440.542480         0.000000e+00 0.000000e+00


In [8]:
print("=" * 60)
print("SUMMED AIC: ALL DATASETS")
print("=" * 60)
print(f"Datasets: {all_datasets}\n")
df_all = summed_aic_table(all_datasets, all_results)
print(df_all.to_string(index=False))

SUMMED AIC: ALL DATASETS
Datasets: ['LohnasKahana2014', 'RepeatedRecallsKahanaJacobs2000']

                 Model    Summed AIC         ΔAIC  Relative Likelihood   AIC Weight
FullReinfPositionalCMR 217126.501587     0.000000         1.000000e+00 1.000000e+00
    BlendPositionalCMR 217199.875366    73.373779         1.167041e-16 1.167041e-16
     BasePositionalCMR 217238.965942   112.464355         3.790317e-25 3.790317e-25
 MCFReinfPositionalCMR 217307.758057   181.256470         4.371774e-40 4.371774e-40
              WeirdCMR 221111.107910  3984.606323         0.000000e+00 0.000000e+00
        NoReinstateCMR 221280.052490  4153.550903         0.000000e+00 0.000000e+00
   DistinctContextsCMR 229720.485962 12593.984375         0.000000e+00 0.000000e+00


## Method 2: Rank-Based Aggregation

In [9]:
def compute_ranks_per_dataset(results_dict):
    """Compute AIC-based ranks for each model within each dataset."""
    ranks = {}
    for data_name in all_datasets:
        # Get mean AIC per model for this dataset
        model_aic = {
            model_title: get_mean_aic(results_dict[data_name][model_title])
            for model_title in model_titles
        }
        # Convert to ranks (1 = best)
        sorted_models = sorted(model_aic.keys(), key=lambda x: model_aic[x])
        ranks[data_name] = {model: rank + 1 for rank, model in enumerate(sorted_models)}
    return ranks

def rank_aggregation_table(results_dict):
    """Create a table showing ranks across datasets and mean rank."""
    ranks = compute_ranks_per_dataset(results_dict)

    data = []
    for model_title in model_titles:
        row = {"Model": model_title}
        for data_name in all_datasets:
            short_name = data_name.replace("RepeatedRecalls", "").replace("Kahana", "K").replace("Gordon", "G").replace("Ranschburg", "R")
            row[short_name] = ranks[data_name][model_title]
        row["Mean Rank"] = np.mean([ranks[dn][model_title] for dn in all_datasets])
        row["Mean Rank (FR)"] = np.mean([ranks[dn][model_title] for dn in free_recall_datasets])
        row["Mean Rank (SR)"] = np.mean([ranks[dn][model_title] for dn in serial_recall_datasets])
        data.append(row)

    df = pd.DataFrame(data)
    df = df.sort_values("Mean Rank").reset_index(drop=True)
    return df

In [10]:
print("=" * 60)
print("RANK-BASED AGGREGATION")
print("=" * 60)
df_ranks = rank_aggregation_table(all_results)
print(df_ranks.to_string(index=False))

RANK-BASED AGGREGATION
                 Model  LohnasK2014  KJacobs2000  Mean Rank  Mean Rank (FR)  Mean Rank (SR)
FullReinfPositionalCMR            1            1        1.0             1.0             1.0
    BlendPositionalCMR            2            3        2.5             2.0             3.0
     BasePositionalCMR            4            2        3.0             4.0             2.0
 MCFReinfPositionalCMR            3            4        3.5             3.0             4.0
              WeirdCMR            6            6        6.0             6.0             6.0
        NoReinstateCMR            7            5        6.0             7.0             5.0
   DistinctContextsCMR            5            7        6.0             5.0             7.0


## Method 3: Meta-Analytic Pooling of ΔAIC

In [11]:
def compute_pairwise_delta_aic_stats(results_dict, data_name, ref_model):
    """Compute mean ΔAIC and SE for each model vs reference model."""
    ref_aic = get_aic_per_subject(results_dict[data_name][ref_model])

    stats_dict = {}
    for model_title in model_titles:
        if model_title == ref_model:
            stats_dict[model_title] = {"mean": 0, "se": 0, "n": len(ref_aic)}
            continue
        model_aic = get_aic_per_subject(results_dict[data_name][model_title])
        delta = model_aic - ref_aic  # positive = model is worse
        stats_dict[model_title] = {
            "mean": np.mean(delta),
            "se": np.std(delta, ddof=1) / np.sqrt(len(delta)),
            "n": len(delta)
        }
    return stats_dict

def meta_analytic_pooling(results_dict, datasets, ref_model):
    """Pool ΔAIC estimates across datasets using inverse-variance weighting."""
    pooled = {}

    for model_title in model_titles:
        if model_title == ref_model:
            pooled[model_title] = {"pooled_mean": 0, "pooled_se": 0, "ci_lower": 0, "ci_upper": 0}
            continue

        means = []
        weights = []

        for data_name in datasets:
            stats = compute_pairwise_delta_aic_stats(results_dict, data_name, ref_model)
            if stats[model_title]["se"] > 0:
                means.append(stats[model_title]["mean"])
                weights.append(1 / (stats[model_title]["se"] ** 2))

        if len(weights) > 0:
            weights = np.array(weights)
            means = np.array(means)
            pooled_mean = np.sum(weights * means) / np.sum(weights)
            pooled_se = np.sqrt(1 / np.sum(weights))
            ci_lower = pooled_mean - 1.96 * pooled_se
            ci_upper = pooled_mean + 1.96 * pooled_se
        else:
            pooled_mean = pooled_se = ci_lower = ci_upper = np.nan

        pooled[model_title] = {
            "pooled_mean": pooled_mean,
            "pooled_se": pooled_se,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper
        }

    return pooled

def meta_analysis_table(results_dict, datasets, label=""):
    """Create meta-analysis table with best model as reference."""
    # First find the best model by summed AIC
    summed = compute_summed_aic(datasets, results_dict)
    best_model = min(summed.keys(), key=lambda x: summed[x])

    pooled = meta_analytic_pooling(results_dict, datasets, best_model)

    data = []
    for model_title in model_titles:
        p = pooled[model_title]
        reliable = "Yes" if p["ci_lower"] > 0 or p["ci_upper"] < 0 else "No"
        if model_title == best_model:
            reliable = "-"
        data.append({
            "Model": model_title,
            "Pooled ΔAIC": f"{p['pooled_mean']:.2f}",
            "95% CI": f"[{p['ci_lower']:.2f}, {p['ci_upper']:.2f}]",
            "Reliably Worse?": reliable
        })

    df = pd.DataFrame(data)
    # Sort by pooled ΔAIC
    df["_sort"] = [pooled[m]["pooled_mean"] for m in df["Model"]]
    df = df.sort_values("_sort").drop("_sort", axis=1).reset_index(drop=True)

    print(f"Reference model (best by summed AIC): {best_model}\n")
    return df

In [12]:
print("=" * 60)
print("META-ANALYTIC POOLING: FREE RECALL")
print("=" * 60)
df_meta_fr = meta_analysis_table(all_results, free_recall_datasets)
print(df_meta_fr.to_string(index=False))

META-ANALYTIC POOLING: FREE RECALL
Reference model (best by summed AIC): FullReinfPositionalCMR

                 Model Pooled ΔAIC         95% CI Reliably Worse?
FullReinfPositionalCMR        0.00   [0.00, 0.00]               -
    BlendPositionalCMR        0.85  [-3.90, 5.59]              No
 MCFReinfPositionalCMR        2.02  [-1.79, 5.83]              No
     BasePositionalCMR        3.21  [-1.55, 7.97]              No
   DistinctContextsCMR        4.38  [-1.17, 9.94]              No
              WeirdCMR       16.01  [8.33, 23.69]             Yes
        NoReinstateCMR       22.41 [12.95, 31.86]             Yes


In [13]:
print("=" * 60)
print("META-ANALYTIC POOLING: SERIAL RECALL")
print("=" * 60)
df_meta_sr = meta_analysis_table(all_results, serial_recall_datasets)
print(df_meta_sr.to_string(index=False))

META-ANALYTIC POOLING: SERIAL RECALL
Reference model (best by summed AIC): FullReinfPositionalCMR

                 Model Pooled ΔAIC           95% CI Reliably Worse?
FullReinfPositionalCMR        0.00     [0.00, 0.00]               -
     BasePositionalCMR        0.01    [-7.62, 7.64]              No
    BlendPositionalCMR        2.30   [-6.27, 10.88]              No
 MCFReinfPositionalCMR        5.82   [-1.82, 13.45]              No
        NoReinstateCMR      177.34 [138.94, 215.73]             Yes
              WeirdCMR      180.23 [143.35, 217.10]             Yes
   DistinctContextsCMR      654.77 [518.96, 790.57]             Yes


In [14]:
print("=" * 60)
print("META-ANALYTIC POOLING: ALL DATASETS")
print("=" * 60)
df_meta_all = meta_analysis_table(all_results, all_datasets)
print(df_meta_all.to_string(index=False))

META-ANALYTIC POOLING: ALL DATASETS
Reference model (best by summed AIC): FullReinfPositionalCMR

                 Model Pooled ΔAIC         95% CI Reliably Worse?
FullReinfPositionalCMR        0.00   [0.00, 0.00]               -
    BlendPositionalCMR        1.19  [-2.97, 5.34]              No
     BasePositionalCMR        2.31  [-1.73, 6.35]              No
 MCFReinfPositionalCMR        2.78  [-0.63, 6.19]              No
   DistinctContextsCMR        5.47 [-0.08, 11.02]              No
              WeirdCMR       22.83 [15.31, 30.34]             Yes
        NoReinstateCMR       31.26 [22.08, 40.44]             Yes


## Summary

In [15]:
print("=" * 60)
print("SUMMARY: BEST MODELS BY POOLING METHOD")
print("=" * 60)
print()
print("FREE RECALL:")
print(f"  Summed AIC winner:     {df_free.iloc[0]['Model']}")
print(f"  Mean rank winner:      {df_ranks.sort_values('Mean Rank (FR)').iloc[0]['Model']}")
print()
print("SERIAL RECALL:")
print(f"  Summed AIC winner:     {df_serial.iloc[0]['Model']}")
print(f"  Mean rank winner:      {df_ranks.sort_values('Mean Rank (SR)').iloc[0]['Model']}")
print()
print("ALL DATASETS:")
print(f"  Summed AIC winner:     {df_all.iloc[0]['Model']}")
print(f"  Mean rank winner:      {df_ranks.sort_values('Mean Rank').iloc[0]['Model']}")

SUMMARY: BEST MODELS BY POOLING METHOD

FREE RECALL:
  Summed AIC winner:     FullReinfPositionalCMR
  Mean rank winner:      FullReinfPositionalCMR

SERIAL RECALL:
  Summed AIC winner:     FullReinfPositionalCMR
  Mean rank winner:      FullReinfPositionalCMR

ALL DATASETS:
  Summed AIC winner:     FullReinfPositionalCMR
  Mean rank winner:      FullReinfPositionalCMR


| Model                        |  Pooled ΔAIC vs Best: mean [CI] | Reliably Worse? |
|---:|:-----------------------------|-------------:|
| Instance-CMR + Trace Boost | 0.00 [0.00, 0.00] |  |
| Instance-CMR | [-1.73, 6.35]  | No |
| Standard CMR | 16.01 [8.33, 23.69]  | Yes |

| Model                        |  Pooled ΔAIC vs Best: mean [CI] | Reliably Worse? |
|---:|:-----------------------------|-------------:|
| Instance-CMR + Trace Boost | 0.00 [0.00, 0.00] |  |
| Instance-CMR | [-1.73, 6.35]  | No |
| Standard CMR | 16.01 [8.33, 23.69]  | Yes |